In [1]:
import json
import os
from pathlib import Path
from typing import Dict, List, Optional
from datetime import datetime

In [2]:
# === Project Root ===
PROJECT_ROOT = Path.cwd().parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "CVEListV5" / "cves"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Metadata file to track processed years
METADATA_FILE = DATA_PROCESSED_DIR / "metadata.json"

print("Raw CVE directory:", DATA_RAW_DIR)
print("Processed data directory:", DATA_PROCESSED_DIR)

Raw CVE directory: c:\Users\kushl\OneDrive\CyberSecRAG\data\raw\CVEListV5\cves
Processed data directory: c:\Users\kushl\OneDrive\CyberSecRAG\data\processed


In [3]:
def load_metadata() -> Dict:
    if METADATA_FILE.exists():
        with open(METADATA_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"processed_years": {}}


def save_metadata(metadata: Dict):
    with open(METADATA_FILE, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

In [4]:
def find_cve_files(year: int) -> List[Path]:
    """
    Returns a list of all CVE JSON files for a given year.
    
    Example:
        find_cve_files(2024)
    """
    year_path = DATA_RAW_DIR / str(year)

    if not year_path.exists():
        raise ValueError(f"Year folder not found: {year_path}")

    # Recursively collect all JSON files
    files = list(year_path.rglob("*.json"))

    return sorted(files)

In [5]:
files_2024 = find_cve_files(2024)
print("Total files:", len(files_2024))
files_2024[:3]

Total files: 38986


[WindowsPath('c:/Users/kushl/OneDrive/CyberSecRAG/data/raw/CVEListV5/cves/2024/0xxx/CVE-2024-0001.json'),
 WindowsPath('c:/Users/kushl/OneDrive/CyberSecRAG/data/raw/CVEListV5/cves/2024/0xxx/CVE-2024-0002.json'),
 WindowsPath('c:/Users/kushl/OneDrive/CyberSecRAG/data/raw/CVEListV5/cves/2024/0xxx/CVE-2024-0003.json')]

In [6]:
from typing import Dict, Optional

def normalize_text(text: str) -> str:
    """
    Cleans and normalizes text for embedding.
    - Removes excessive whitespace
    - Strips leading/trailing spaces
    """
    if not text:
        return ""

    return " ".join(text.strip().split())


def extract_cve_record(data: Dict, year: int) -> Optional[Dict]:
    """
    Extracts structured, ML-ready fields from a raw CVE JSON record.
    Returns a clean dictionary or None if critical data is missing.
    """

    try:
        # --- CVE ID ---
        cve_id = data.get("cveMetadata", {}).get("cveId")
        if not cve_id:
            return None

        cna = data.get("containers", {}).get("cna", {})

        # --- Description ---
        descriptions = cna.get("descriptions", [])
        description = ""
        for d in descriptions:
            if d.get("lang") == "en":
                description = d.get("value", "")
                break

        description = normalize_text(description)
        if not description:
            return None  # critical field

        # --- CVSS (prioritize v3.1 > v3.0 > fallback None) ---
        cvss_score = None
        severity = ""

        metrics = cna.get("metrics", [])
        for metric in metrics:
            if "cvssV3_1" in metric:
                cvss_score = metric["cvssV3_1"].get("baseScore")
                severity = metric["cvssV3_1"].get("baseSeverity", "")
                break
            elif "cvssV3_0" in metric:
                cvss_score = metric["cvssV3_0"].get("baseScore")
                severity = metric["cvssV3_0"].get("baseSeverity", "")
                break

        # --- Affected Products (keep flat for minimal change) ---
        affected_entries = cna.get("affected", [])
        affected_products = []

        for entry in affected_entries:
            vendor = entry.get("vendor")
            product = entry.get("product")

            if vendor and product:
                affected_products.append(f"{vendor} {product}")

        affected_products = list(set(affected_products))  # remove duplicates

        # --- CWE + Problem Type ---
        cwe_id = None
        problem_type = ""

        problem_types = cna.get("problemTypes", [])
        for pt in problem_types:
            descriptions = pt.get("descriptions", [])
            for desc in descriptions:
                if desc.get("cweId"):
                    cwe_id = desc.get("cweId")
                    problem_type = normalize_text(desc.get("description", ""))
                    break

        # --- Build Embedding Text (NEW ADDITION) ---
        embedding_parts = [
            f"CVE ID: {cve_id}",
            f"Year: {year}",
            f"Severity: {severity}" if severity else "",
            f"CVSS Score: {cvss_score}" if cvss_score is not None else "",
            f"Weakness Type: {cwe_id}" if cwe_id else "",
            f"Problem Type: {problem_type}" if problem_type else "",
            f"Affected Products: {', '.join(affected_products)}" if affected_products else "",
            f"Description: {description}"
        ]

        embedding_text = "\n".join([p for p in embedding_parts if p])

        # --- Final Structured Record ---
        record = {
            "cve_id": cve_id,
            "year": year,
            "description": description,
            "cvss_score": cvss_score,
            "severity": severity,
            "affected_products": affected_products,
            "cwe_id": cwe_id,
            "problem_type": problem_type,          # NEW
            "embedding_text": embedding_text       # NEW
        }

        return record

    except Exception:
        return None

In [7]:
sample_file = files_2024[0]

with open(sample_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

record = extract_cve_record(raw_data, 2024)
record

{'cve_id': 'CVE-2024-0001',
 'year': 2024,
 'description': 'A condition exists in FlashArray Purity whereby a local account intended for initial array configuration remains active potentially allowing a malicious actor to gain elevated privileges.',
 'cvss_score': 10,
 'severity': 'CRITICAL',
 'affected_products': ['Pure Storage FlashArray'],
 'cwe_id': 'CWE-1188',
 'problem_type': 'CWE-1188 Insecure Default Initialization of Resource',
 'embedding_text': 'CVE ID: CVE-2024-0001\nYear: 2024\nSeverity: CRITICAL\nCVSS Score: 10\nWeakness Type: CWE-1188\nProblem Type: CWE-1188 Insecure Default Initialization of Resource\nAffected Products: Pure Storage FlashArray\nDescription: A condition exists in FlashArray Purity whereby a local account intended for initial array configuration remains active potentially allowing a malicious actor to gain elevated privileges.'}

In [8]:
# %%
from datetime import datetime, timezone
def process_year(year: int):
    """
    Process all CVE files for a given year.
    Saves structured records as JSONL.
    Uses metadata.json to prevent reprocessing.
    """

    metadata = load_metadata()

    if str(year) in metadata["processed_years"]:
        print(f"[✓] Year {year} already processed. Skipping.")
        return

    files = find_cve_files(year)

    output_file = DATA_PROCESSED_DIR / f"{year}.jsonl"

    total_files = 0
    valid_records = 0

    print(f"\nProcessing year {year}...")
    print(f"Total files found: {len(files)}")

    with open(output_file, "w", encoding="utf-8") as out:

        for file_path in files:
            total_files += 1

            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                record = extract_cve_record(data, year)

                if record:
                    out.write(json.dumps(record) + "\n")
                    valid_records += 1

            except Exception:
                continue

    # Update metadata
    metadata["processed_years"][str(year)] = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "total_files": total_files,
        "valid_records": valid_records
    }

    save_metadata(metadata)

    print(f"\nYear {year} complete.")
    print(f"Valid records: {valid_records}")
    print(f"Output saved to: {output_file}")

In [9]:
#sample year , checking if the function is working all well
process_year(2024)

[✓] Year 2024 already processed. Skipping.
